# SegFormer-B0 defect segmentation (Kaggle)

**Before running:** Settings -> Accelerator = GPU T4 x2 or P100, Internet = ON (needed for `nvidia/mit-b0`).
Add as inputs: your segmentation datasets + a dataset containing `train_segformer.py` and `prepare_data.py`.

In [ ]:
!nvidia-smi
!ls /kaggle/input

In [ ]:
!pip install -q transformers albumentations

In [ ]:
# ---- CONFIG: edit these after checking `ls /kaggle/input` ----
SCRIPTS = "/kaggle/input/defect-scripts"          # dataset holding the two .py files
SOURCES = {                                        # name: (images_dir, masks_dir)
    "crackseg9k": ("/kaggle/input/crackseg9k/images", "/kaggle/input/crackseg9k/masks"),
    # "deepcrack": ("/kaggle/input/deepcrack/train_img", "/kaggle/input/deepcrack/train_lab"),
}
DATA = "/kaggle/working/data"
RUN  = "/kaggle/working/runs/b0_v1"
EPOCHS, BS, LR = 50, 16, 6e-5   # BS is PER GPU (T4x2 -> effective 32)


In [ ]:
import os, shutil, subprocess
for f in ("train_segformer.py", "prepare_data.py"):
    shutil.copy(f"{SCRIPTS}/{f}", f"/kaggle/working/{f}")
os.chdir("/kaggle/working")

if not os.path.isdir(f"{DATA}/train/images"):
    cmd = ["python", "prepare_data.py", "--out", DATA, "--max_side", "1024", "--val_frac", "0.1"]
    for n, (i, m) in SOURCES.items():
        cmd += ["--src", f"{n}={i}:{m}"]
    subprocess.run(cmd, check=True)
else:
    print("data already prepared")


In [ ]:
# Resumes automatically if last.pt exists (after the 12h limit: attach previous output as input, copy last.pt into RUN)
# torchrun = DDP across both T4s. workers=2 per process because Kaggle only gives ~4 CPU cores.
resume = f"--resume {RUN}/last.pt" if os.path.exists(f"{RUN}/last.pt") else ""
!torchrun --nproc_per_node=2 train_segformer.py --data_root {DATA} --out_dir {RUN} --img_size 512 \
    --batch_size {BS} --epochs {EPOCHS} --lr {LR} --workers 2 {resume}
# If NCCL hangs/errors on init, rerun with:  !NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 torchrun ...


In [ ]:
import json
m = json.load(open(f"{RUN}/metrics.json"))
best = max(m, key=lambda x: x["mIoU"])
print("best epoch:", best["epoch"], {k: round(v, 4) for k, v in best.items() if k != "epoch"})


In [ ]:
# Package best model for download (Output tab) -> use with from_pretrained() locally / in ROS node
shutil.make_archive("/kaggle/working/segformer_b0_best", "zip", f"{RUN}/best")
